# TN1 đối chứng — LSTM thu nhỏ về cùng ngân sách tham số

## Câu hỏi

TN1 cho kết quả rõ ràng, đủ ba seed:

| cấu hình | tham số | cv_mean | seed_std |
|---|---|---|---|
| **LSTM-352** | 1.502.713 | **0.756997** | 0.004141 |
| TCN-64 | 151.513 | 0.742338 | 0.004369 |
| DS-TCN-64 | 56.281 | 0.742110 | 0.000673 |

LSTM hơn **0.0149**, gấp 3,4 lần độ lệch seed. Ba dải seed không chồng nhau, nên
khoảng cách này là thật.

Câu hỏi ngay sau đó:

> **LSTM hơn vì kiến trúc hồi quy, hay chỉ vì nó to gấp 26 lần?**

## Cách trả lời

Thu LSTM về đúng ngân sách tham số của DS-TCN-64:

```
LSTM  hidden=352   1.502.713 tham số
LSTM  hidden=67       56.908 tham số     <- notebook này
DS-TCN-64             56.281 tham số
```

Chỉ đổi `hidden`. Số lớp 2, độ dài dự báo 25, và toàn bộ cấu hình huấn luyện
giữ nguyên.

Hai kết quả đều dùng được:

| LSTM-67 ra | kết luận |
|---|---|
| tụt hẳn, ví dụ 0.68 | **hiệu quả đến từ kiến trúc.** Ở ngân sách nhỏ, tích chập giữ được điểm còn hồi quy thì không |
| bám sát 0.742 | **bài toán không cần model to.** LSTM-352 thừa dung lượng, và TCN không đóng góp gì riêng |

## Giới hạn của phép so này

Bộ tham số huấn luyện lấy từ `checkpoints/optimal_params.json` của MobiVital —
lr `1e-4`, 20 epoch, batch 64 — được dò cho `hidden=352`. LSTM-67 chạy bằng bộ
đó tức là dùng cấu hình huấn luyện của một model khác hẳn kích thước, chưa được
dò riêng.

Điều kiện này **áp dụng như nhau cho cả hai bên**: DS-TCN-64 cũng dùng đúng bộ
tham số đó và cũng không được dò. Vì vậy phép so LSTM-67 với DS-TCN-64 ở cùng
ngân sách tham số vẫn có giá trị — chênh lệch quan sát được không đến từ việc
một kiến trúc được ưu ái hơn kiến trúc kia.

Ngược lại, so LSTM-67 với LSTM-352 rồi kết luận về năng lực của LSTM cỡ nhỏ thì
không hợp lệ, vì chỉ LSTM-352 có tham số huấn luyện dò riêng cho nó. Đó cũng
không phải câu hỏi mà thí nghiệm này đặt ra.

Nếu LSTM-67 và DS-TCN-64 cho kết quả sát nhau, cần thêm một vòng dò `lr` cho cả
hai kiến trúc mới phân định được.

## 1. Chuẩn bị Colab

**Chọn T4, không cần L4.** Hidden 67 nhỏ hơn 352 hơn năm lần nên bước chấm điểm
nhẹ hẳn — T4 15 GB thoải mái, và tốn ít compute unit hơn.

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. LSTM-67 — 4 fold CV, 3 seed

Ba seed để so được với ba seed của các cấu hình khác. So `mean` của ba seed với
một con số đơn là so hai đại lượng khác nhau.

Cấu hình huấn luyện giữ y nguyên như ba cấu hình trước: 20 epoch, Adam lr 1e-4,
batch 64, MSE, `corr` 0.9, bốn fold cũ. Chỉ đổi `--hidden 67`.

Tên cấu hình là `lstm_h67_mse_corr0.9_seed<N>`. Hậu tố `_h67` chỉ xuất hiện khi
`hidden` khác 352, nên mọi tên của TN0 và TN1 giữ nguyên.

Sau mỗi fold script tự nén rồi chép sang Drive, tên tệp chứa cấu hình nên không
đè tệp nào. Phiên bị ngắt giữa chừng thì chạy lại ô này, các fold đã xong được
bỏ qua.

Khoảng **2 giờ** trên T4.

In [ ]:
!python scripts/run_cv.py --experiment tn1 --model lstm --hidden 67 --seed 0
!python scripts/run_cv.py --experiment tn1 --model lstm --hidden 67 --seed 1
!python scripts/run_cv.py --experiment tn1 --model lstm --hidden 67 --seed 2

## 3. Xem kết quả

Phiên này chỉ có bốn dòng của LSTM-67 trong `summary.csv`. Bảng đủ bốn cấu hình
dựng ở `TN1_final_evaluation.ipynb` sau khi gộp các tệp nén.

Ô này chạy trong vài giây, bấm lại bao nhiêu lần cũng được.

In [ ]:
!python scripts/compare_cv.py --experiment tn1

## 4. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Ô này đóng phiên
lại. Kết quả đã nén sang Drive sau mỗi fold nên ngắt ở đây không mất gì.

Ba ô ở mục 2, 3, 4 bấm liên tiếp thì Colab xếp hàng chạy lần lượt: train xong
tới in bảng, in xong tới ngắt. Không cần ngồi canh.

Chạy ô này là mất kết nối, muốn dùng tiếp phải bấm Connect và chạy lại mục 1.

In [ ]:
from google.colab import runtime
runtime.unassign()